In [4]:
from google.colab import userdata
from google import genai
from google.genai import types
from typing import TypedDict, List, Optional
import json
import re

#Secret api key
API_KEY = userdata.get("Gen_Ai")
client = genai.Client(api_key=API_KEY)

#Model
MODEL_NAME = ("gemini-2.5-flash-lite")

#Scema

class Education(TypedDict):
    Degree: Optional[str]
    Institute: Optional[str]
    Year: Optional[str]
    Cgpa: Optional[str]


class Experience(TypedDict):
    Company: Optional[str]
    Role: Optional[str]
    Duration: Optional[str]
    Highlights: List[str]


class Project(TypedDict):
    Project: Optional[str]
    Description: Optional[str]
    Tech_stack: List[str]


class Resume(TypedDict):
    Name: Optional[str]
    Email: Optional[str]
    Phone: Optional[str]
    Linkedin_url: Optional[str]
    Education: List[Education]
    Skills: List[str]
    Experience: List[Experience]
    Projects: List[Project]
    Total_experience_years: float
    Summary: Optional[str]



#SystemInstructions


SYSTEM_INSTRUCTION = """
You are an expert HR resume parser.

Rules:

1. Extract only information present in the resume.
2. Do not invent values.
3. Use null for missing fields.
4. Output must be valid JSON only.
5. Skills should be clean and unique.
6. Normalize phone numbers to +91XXXXXXXXXX if possible.
7. Summary must be one sentence.
8. Resume can be in English or Any Indian languages.
9. Output values must always be in English.
"""

# RESPONSE

RESPONSE = {
    "type": "OBJECT",
    "properties": {

        "Name": {
            "type": "STRING"
        },

        "Email": {
            "type": "STRING"
        },

        "Phone": {
            "type": "STRING"
        },

        "Linkedin_url": {
            "type": "STRING"
        },

        "Education": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {

                    "Name of the degree": {
                        "type": "STRING"
                    },

                    "Name of the institute": {
                        "type": "STRING"
                    },

                    "Year of study": {
                        "type": "STRING"
                    },

                    "Cgpa": {
                        "type": "STRING"
                    }
                }
            }
        },

        "Skills": {
            "type": "ARRAY",
            "items": {
                "type": "STRING"
            }
        },

        "Experience": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {

                    "Name of the company": {
                        "type": "STRING"
                    },

                    "Role": {
                        "type": "STRING"
                    },

                    "Duration": {
                        "type": "STRING"
                    },

                    "Highlights": {
                        "type": "ARRAY",
                        "items": {
                            "type": "STRING"
                        }
                    }
                }
            }
        },

        "Projects": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {

                    "Name": {
                        "type": "STRING"
                    },

                    "Description": {
                        "type": "STRING"
                    },

                    "Tech_stack": {
                        "type": "ARRAY",
                        "items": {
                            "type": "STRING"
                        }
                    }
                }
            }
        },

        "Total_experience_years": {
            "type": "NUMBER"
        },

        "Summary": {
            "type": "STRING"
        }
    }
}

#Validation


def validate_email(email):

    if email is None:
        return True

    return "@" in email


def validate_year(year):

    if year is None:
        return True

    match = re.findall(r"\d{4}", str(year))

    if not match:
        return True

    year_num = int(match[0])

    return 1990 <= year_num <= 2026


def validate_experience(exp):

    try:
        return float(exp) >= 0

    except:
        return False

# Parse resume

import time

def parse_resume(text: str) -> dict:

    prompt = f"""
    Parse this resume and return JSON.

    Resume:
    {text}
    """

    for attempt in range(3):

        try:

            response = client.models.generate_content(

                model=MODEL_NAME,

                contents=prompt,

                config=types.GenerateContentConfig(

                    system_instruction=INSTRUCTION,

                    response_mime_type="application/json",

                    response_schema=RESPONSE,

                    temperature=0.2,

                    max_output_tokens=2048
                )
            )

            data = json.loads(response.text)

            # Email check
            if not validate_email(data.get("email")):
                data["email"] = None

            # Year check
            for edu in data.get("education", []):

                if not validate_year(edu.get("year")):
                    edu["year"] = None

            # Experience check
            if not validate_experience(
                data.get("total_experience_years", 0)
            ):
                data["total_experience_years"] = 0.0

            return data

        except Exception as e:

            print(f"Retry {attempt + 1}...")

            time.sleep(5)

    return {
        "error": "Model unavailable"
    }
#Test-1 Prompt Only
def parse_prompt_only(text):

    prompt = f"""
    Extract resume information and return JSON.

    Resume:
    {text}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text

#Test-2 — MimeType
def parse_mime_type(text):

    prompt = f"""
    Extract resume information and return JSON.

    Resume:
    {text}
    """

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(

            response_mime_type="application/json",

            temperature=0.2
        )
    )

    return response.text

#Test-3 — Response schema
def parse_schema(text):

    prompt = f"""
    Extract resume information.

    Resume:
    {text}
    """

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=INSTRUCTION,

            response_mime_type="application/json",

            response_schema=RESPONSE,

            temperature=0.2,

            max_output_tokens=2048
        )
    )

    return response.text


#TEST
resume_text = """
Name: Rohit Sharma
Email: Rohit45@gmail.com
Phone: 9826443264

Skills:
Python, Java, A.I

Experience:
Freasher

Projects:
Data analytics with Tableau
"""


print("Test-1 Prompt Only")

output1 = parse_prompt_only(resume_text)

print(output1)


print("\nTest-2 — MimeType")

output2 = parse_mime_type(resume_text)

print(output2)


print("\nTest-3 — Response schema")

output3 = parse_schema(resume_text)

print(output3)


parsed = parse_resume(resume_text)

print(json.dumps(parsed, indent=2, ensure_ascii=False))

Test-1 Prompt Only


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 47.620389106s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-fla